# AI-Based Crop-Safe Path Planning for an Autonomous Potato-Field Robot


---

## Overview
This standalone explanatory notebook implements all **6 assignment tasks** for guiding an autonomous monitoring rover across a 2D potato field:

1. **Task 1 — Field Environment:** 2D grid modeling safe lanes, potato crop rows, crop-protection regions, static obstacles, unknown obstacles, start depot, and monitoring destination.
2. **Task 2 — Problem Formulation:** Explicit initial state, 4-directional actions, deterministic transition model, goal test, path-cost function, and admissible Manhattan heuristic.
3. **Task 3 — Search Algorithms:** Breadth-First Search (BFS), Depth-First Search (DFS), Greedy Best-First Search (GBFS), and A* Search.
4. **Task 4 — Crop-Safe Planning:** Hard exclusion of protected crops/obstacles plus soft proximity penalties on adjacent lanes.
5. **Task 5 — Online Obstacle Detection & Replanning:** Simulated sensor sweep discovering hidden obstacles mid-mission, updating the robot's internal map, and triggering A* replanning.
6. **Task 6 — Performance Evaluation:** Empirical comparison table (path length, cost, risk steps, expanded nodes, runtime, goal status, replans) and visual plots.


In [ ]:
from collections import deque
import heapq
import time
from typing import Dict, List, Optional, Set, Tuple

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np

# Cell Type Encodings
SAFE = 0
POTATO_ROW = 1
PROTECTED = 2
OBSTACLE = 3
UNKNOWN_OBSTACLE = 4
START = 5
GOAL = 6

# 4-Directional Cardinal Actions: Up, Right, Down, Left
DIRS: List[Tuple[int, int]] = [(-1, 0), (0, 1), (1, 0), (0, -1)]
DIR_NAMES: List[str] = ["Up", "Right", "Down", "Left"]
Coord = Tuple[int, int]

## Task 1 — 2D Potato-Field Environment

We model the field as a discrete `12 x 16` NumPy matrix:
- **Safe Travelling Cells (`.` / `0`)**: Tilled soil lanes between crop ridges where the robot can drive.
- **Potato Rows (`P` / `1`)**: Planted crop ridges that the rover must travel *between* without crushing.
- **Crop-Protection Regions (`C` / `2`)**: Delicate seedling beds; strictly impassable, and adjacent lanes incur a safety penalty.
- **Static Obstacles (`#` / `3`)**: Known stones or farm tools.
- **Unknown Obstacles (`?` / `4`)**: Hidden obstacles unknown at mission start (used in Task 5).
- **Start (`S` / `5`) & Goal (`G` / `6`)**: Rover depot and monitoring destination.

In [ ]:
def build_default_field() -> Tuple[np.ndarray, Coord, Coord]:
    layout = [
        "S..P......P....P",
        ".#.P.PPPP.P.PP.P",
        "...P......P....P",
        ".PP..CCCC..#PP..",
        ".....CCCC..?....",
        ".PPP.CCCC.PPPP..",
        ".....#..........",
        ".PPP?.PPP..PPP..",
        ".....P...P...P..",
        ".P...P.#.P...P.#",
        ".P.PPP...PPP.P..",
        ".......#....?..G",
    ]
    char_to_code = {
        ".": SAFE, "P": POTATO_ROW, "C": PROTECTED,
        "#": OBSTACLE, "?": UNKNOWN_OBSTACLE, "S": START, "G": GOAL,
    }
    grid = np.array([[char_to_code[ch] for ch in row] for row in layout], dtype=int)
    start = tuple(map(int, np.argwhere(grid == START)[0]))
    goal = tuple(map(int, np.argwhere(grid == GOAL)[0]))
    return grid, start, goal

def render_grid_on_axis(
    ax: plt.Axes,
    grid: np.ndarray,
    title: str,
    path: Optional[List[Coord]] = None,
    secondary_path: Optional[List[Coord]] = None,
    hide_unknown: bool = False,
) -> None:
    rows, cols = grid.shape
    color_map = {
        SAFE: "#26211c",
        POTATO_ROW: "#2f6b2f",
        PROTECTED: "#0f766e",
        OBSTACLE: "#475569",
        UNKNOWN_OBSTACLE: "#26211c" if hide_unknown else "#d97706",
        START: "#0ea5e9",
        GOAL: "#eab308",
    }
    for r in range(rows):
        for c in range(cols):
            code = int(grid[r, c])
            rect = patches.Rectangle(
                (c, rows - 1 - r), 1, 1,
                facecolor=color_map[code], edgecolor="#111827", linewidth=0.8
            )
            ax.add_patch(rect)
            if code == PROTECTED:
                ax.text(c + 0.5, rows - 1 - r + 0.5, "C", ha="center", va="center", color="#ccfbf1", fontsize=7, fontweight="bold")
            elif code == UNKNOWN_OBSTACLE and not hide_unknown:
                ax.text(c + 0.5, rows - 1 - r + 0.5, "?", ha="center", va="center", color="#fffbeb", fontsize=8, fontweight="bold")
    if secondary_path and len(secondary_path) > 1:
        ax.plot([c + 0.5 for _, c in secondary_path], [rows - 1 - r + 0.5 for r, _ in secondary_path], color="#fb7185", linewidth=2.0, linestyle="--", label="Initial / Standard")
    if path and len(path) > 1:
        ax.plot([c + 0.5 for _, c in path], [rows - 1 - r + 0.5 for r, _ in path], color="#34d399", linewidth=2.6, marker="o", markersize=3, label="Planned / Traveled")
    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=11, fontweight="bold")

grid, start, goal = build_default_field()
fig, ax = plt.subplots(figsize=(8, 6))
render_grid_on_axis(ax, grid, f"Task 1: Potato Field Grid (Start={start}, Goal={goal})")
plt.show()

## Task 2 — Problem Formulation & Heuristic Design

- **Initial State:** $s_0 = (r_{\text{start}}, c_{\text{start}}) = (0, 0)$
- **Available Actions:** 4-directional cardinal movement $A(s) \subseteq \{\text{Up } (-1,0),\; \text{Right } (0,+1),\; \text{Down } (+1,0),\; \text{Left } (0,-1)\}$. Diagonal movement across planted rows is disabled.
- **Transition Model:** Deterministic $\text{Result}(s, a) = (r + \Delta r, c + \Delta c)$ provided the target cell is within grid bounds and passable.
- **Goal Test:** $\text{IsGoal}(s) \iff (s == s_{\text{goal}})$.
- **Path-Cost Function:**
  $$c(s, a, s') = 1 + \text{penalty}(s')$$
  where $\text{penalty}(s') = w_{\text{safe}}$ (default `5`) if $s'$ is orthogonally adjacent to a `PROTECTED` cell, and `0` otherwise.
- **Heuristic Function (Informed Search):** Manhattan Distance
  $$h(n) = |r_n - r_{\text{goal}}| + |c_n - c_{\text{goal}}|$$
  Because every grid move has cost $\ge 1$ and no diagonal shortcuts exist, $h(n) \le h^*(n)$ holds everywhere—making $h(n)$ both **admissible** and **consistent**, guaranteeing optimal solutions for A*.

In [ ]:
def manhattan(a: Coord, b: Coord) -> int:
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def is_passable(grid: np.ndarray, r: int, c: int, treat_unknown_as_obstacle: bool) -> bool:
    rows, cols = grid.shape
    if not (0 <= r < rows and 0 <= c < cols):
        return False
    cell = int(grid[r, c])
    if cell in (OBSTACLE, PROTECTED, POTATO_ROW):
        return False
    if treat_unknown_as_obstacle and cell == UNKNOWN_OBSTACLE:
        return False
    return True

def is_near_protected(grid: np.ndarray, r: int, c: int) -> bool:
    rows, cols = grid.shape
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and int(grid[nr, nc]) == PROTECTED:
            return True
    return False

def step_cost(grid: np.ndarray, r: int, c: int, penalty_weight: int) -> int:
    if penalty_weight <= 0 or int(grid[r, c]) in (START, GOAL):
        return 1
    return 1 + (penalty_weight if is_near_protected(grid, r, c) else 0)

def count_risk_steps(grid: np.ndarray, path: List[Coord]) -> int:
    return sum(1 for r, c in path[1:] if is_near_protected(grid, r, c))

def reconstruct_path(parent: Dict[Coord, Coord], start: Coord, goal: Coord) -> List[Coord]:
    path: List[Coord] = []
    curr = goal
    while curr in parent or curr == start:
        path.append(curr)
        if curr == start:
            break
        curr = parent[curr]
    path.reverse()
    return path if path and path[0] == start else []

print(f"Initial State: {start}, Goal State: {goal}, Manhattan h(start) = {manhattan(start, goal)}")

## Task 3 — Search Algorithms

We implement four classical AI search strategies:
1. **Breadth-First Search (BFS)**: Uses a FIFO queue; guarantees shortest path in number of hops on unweighted grids, but ignores crop-safe proximity costs.
2. **Depth-First Search (DFS)**: Uses a LIFO stack; low memory footprint, explores deep corridors first, not optimal.
3. **Greedy Best-First Search (GBFS)**: Orders the frontier purely by $f(n) = h(n)$; reaches the goal quickly with minimal expansions, but ignores accumulated path cost $g(n)$.
4. **A* Search**: Orders the frontier by $f(n) = g(n) + h(n)$, combining exact accumulated cost (including safety penalties) with the admissible Manhattan heuristic.

In [ ]:
def bfs(grid: np.ndarray, start: Coord, goal: Coord, penalty_weight: int = 5) -> dict:
    t0 = time.perf_counter()
    queue = deque([start])
    visited: Set[Coord] = {start}
    parent: Dict[Coord, Coord] = {}
    expanded, reached = 0, False
    while queue:
        curr = queue.popleft()
        expanded += 1
        if curr == goal:
            reached = True
            break
        for dr, dc in DIRS:
            nxt = (curr[0] + dr, curr[1] + dc)
            if nxt not in visited and is_passable(grid, nxt[0], nxt[1], False):
                visited.add(nxt)
                parent[nxt] = curr
                queue.append(nxt)
    elapsed = (time.perf_counter() - t0) * 1000.0
    path = reconstruct_path(parent, start, goal) if reached else []
    cost = sum(step_cost(grid, r, c, penalty_weight) for r, c in path[1:])
    return {"name": "BFS", "path": path, "length": max(0, len(path)-1), "cost": cost, "risk_steps": count_risk_steps(grid, path) if reached else 0, "expanded": expanded, "time_ms": elapsed, "reached": reached}

def dfs(grid: np.ndarray, start: Coord, goal: Coord, penalty_weight: int = 5) -> dict:
    t0 = time.perf_counter()
    stack: List[Coord] = [start]
    visited: Set[Coord] = {start}
    parent: Dict[Coord, Coord] = {}
    expanded, reached = 0, False
    while stack:
        curr = stack.pop()
        expanded += 1
        if curr == goal:
            reached = True
            break
        for dr, dc in reversed(DIRS):
            nxt = (curr[0] + dr, curr[1] + dc)
            if nxt not in visited and is_passable(grid, nxt[0], nxt[1], False):
                visited.add(nxt)
                parent[nxt] = curr
                stack.append(nxt)
    elapsed = (time.perf_counter() - t0) * 1000.0
    path = reconstruct_path(parent, start, goal) if reached else []
    cost = sum(step_cost(grid, r, c, penalty_weight) for r, c in path[1:])
    return {"name": "DFS", "path": path, "length": max(0, len(path)-1), "cost": cost, "risk_steps": count_risk_steps(grid, path) if reached else 0, "expanded": expanded, "time_ms": elapsed, "reached": reached}

def greedy_best_first(grid: np.ndarray, start: Coord, goal: Coord, penalty_weight: int = 5) -> dict:
    t0 = time.perf_counter()
    heap = [(manhattan(start, goal), 0, start)]
    visited: Set[Coord] = {start}
    parent: Dict[Coord, Coord] = {}
    expanded, counter, reached = 0, 1, False
    while heap:
        _, _, curr = heapq.heappop(heap)
        expanded += 1
        if curr == goal:
            reached = True
            break
        for dr, dc in DIRS:
            nxt = (curr[0] + dr, curr[1] + dc)
            if nxt not in visited and is_passable(grid, nxt[0], nxt[1], False):
                visited.add(nxt)
                parent[nxt] = curr
                heapq.heappush(heap, (manhattan(nxt, goal), counter, nxt))
                counter += 1
    elapsed = (time.perf_counter() - t0) * 1000.0
    path = reconstruct_path(parent, start, goal) if reached else []
    cost = sum(step_cost(grid, r, c, penalty_weight) for r, c in path[1:])
    return {"name": "Greedy Best-First", "path": path, "length": max(0, len(path)-1), "cost": cost, "risk_steps": count_risk_steps(grid, path) if reached else 0, "expanded": expanded, "time_ms": elapsed, "reached": reached}

def a_star(grid: np.ndarray, start: Coord, goal: Coord, penalty_weight: int = 5, treat_unknown_as_obstacle: bool = False, label: str = "A* (Crop-Safe)") -> dict:
    t0 = time.perf_counter()
    g_score: Dict[Coord, int] = {start: 0}
    heap = [(manhattan(start, goal), manhattan(start, goal), 0, start)]
    closed: Set[Coord] = set()
    parent: Dict[Coord, Coord] = {}
    expanded, counter, reached = 0, 1, False
    while heap:
        _, _, _, curr = heapq.heappop(heap)
        if curr in closed:
            continue
        closed.add(curr)
        expanded += 1
        if curr == goal:
            reached = True
            break
        for dr, dc in DIRS:
            nxt = (curr[0] + dr, curr[1] + dc)
            if nxt in closed or not is_passable(grid, nxt[0], nxt[1], treat_unknown_as_obstacle):
                continue
            tentative_g = g_score[curr] + step_cost(grid, nxt[0], nxt[1], penalty_weight)
            if nxt not in g_score or tentative_g < g_score[nxt]:
                g_score[nxt] = tentative_g
                parent[nxt] = curr
                h = manhattan(nxt, goal)
                heapq.heappush(heap, (tentative_g + h, h, counter, nxt))
                counter += 1
    elapsed = (time.perf_counter() - t0) * 1000.0
    path = reconstruct_path(parent, start, goal) if reached else []
    return {"name": label, "path": path, "length": max(0, len(path)-1), "cost": g_score.get(goal, 0) if reached else 0, "risk_steps": count_risk_steps(grid, path) if reached else 0, "expanded": expanded, "time_ms": elapsed, "reached": reached}

## Task 4 — Crop-Safe Planning Comparison

We verify that **Standard A*** (`penalty_weight=0`) may hug sensitive `PROTECTED` crop beds to shave off distance, whereas **Crop-Safe A*** (`penalty_weight=5`) routes into safer central tracks whenever available.

In [ ]:
std_a = a_star(grid, start, goal, penalty_weight=0, label="A* (Standard)")
safe_a = a_star(grid, start, goal, penalty_weight=5, label="A* (Crop-Safe)")

print(f"Standard A*  -> Length: {std_a['length']} | Near-Crop Risk Steps: {std_a['risk_steps']}")
print(f"Crop-Safe A* -> Length: {safe_a['length']} | Near-Crop Risk Steps: {safe_a['risk_steps']} | Cost: {safe_a['cost']}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
render_grid_on_axis(ax1, grid, f"Standard A* (Risk Steps: {std_a['risk_steps']})", path=std_a["path"])
render_grid_on_axis(ax2, grid, f"Crop-Safe A* (Risk Steps: {safe_a['risk_steps']})", path=safe_a["path"])
plt.tight_layout()
plt.show()

## Task 5 — Unknown Obstacle Detection & Dynamic A* Replanning

Initially, the robot's `known_grid` masks all `UNKNOWN_OBSTACLE` cells as `SAFE`. During execution:
1. At each step, the robot scans cells within Manhattan `sensor_radius = 2`.
2. When an unknown obstacle is detected, `known_grid` is updated to `OBSTACLE`.
3. If any cell on the remaining planned route is now blocked, the robot immediately invokes **A*** from its current position to `goal`.

In [ ]:
def online_replanning_astar(
    true_grid: np.ndarray, start: Coord, goal: Coord, penalty_weight: int = 5, sensor_radius: int = 2
) -> dict:
    t0 = time.perf_counter()
    known_grid = true_grid.copy()
    known_grid[known_grid == UNKNOWN_OBSTACLE] = SAFE
    rows, cols = true_grid.shape
    initial = a_star(known_grid, start, goal, penalty_weight, True, "Initial Plan")
    initial_path = list(initial["path"])
    route = list(initial_path)
    total_expanded = initial["expanded"]
    replan_count = 0
    robot = start
    traveled = [robot]

    while robot != goal and route:
        newly_detected = []
        for r in range(max(0, robot[0] - sensor_radius), min(rows, robot[0] + sensor_radius + 1)):
            for c in range(max(0, robot[1] - sensor_radius), min(cols, robot[1] + sensor_radius + 1)):
                if abs(r - robot[0]) + abs(c - robot[1]) <= sensor_radius:
                    if int(true_grid[r, c]) == UNKNOWN_OBSTACLE and int(known_grid[r, c]) != OBSTACLE:
                        known_grid[r, c] = OBSTACLE
                        newly_detected.append((r, c))
        idx = route.index(robot) if robot in route else 0
        if newly_detected and any(int(known_grid[r, c]) == OBSTACLE for r, c in route[idx:]):
            replan_count += 1
            replan = a_star(known_grid, robot, goal, penalty_weight, True)
            total_expanded += replan["expanded"]
            print(f"Step {len(traveled)-1}: Detected {newly_detected} from {robot} -> Replan #{replan_count}")
            if not replan["reached"]:
                route = []
                break
            route = replan["path"]
            idx = 0
        if idx + 1 >= len(route):
            break
        robot = route[idx + 1]
        traveled.append(robot)

    elapsed = (time.perf_counter() - t0) * 1000.0
    reached = robot == goal
    cost = sum(step_cost(true_grid, r, c, penalty_weight) for r, c in traveled[1:]) if reached else 0
    return {
        "name": "Online A* Replanning", "path": traveled if reached else [],
        "initial_path": initial_path, "length": max(0, len(traveled)-1) if reached else 0,
        "cost": cost, "risk_steps": count_risk_steps(true_grid, traveled) if reached else 0,
        "expanded": total_expanded, "time_ms": elapsed, "reached": reached, "replans": replan_count,
    }

online_res = online_replanning_astar(grid, start, goal, penalty_weight=5, sensor_radius=2)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
render_grid_on_axis(ax1, grid, "Initial Plan (Unknowns Hidden)", path=online_res["initial_path"], hide_unknown=True)
render_grid_on_axis(ax2, grid, f"Final Replanned Route ({online_res['replans']} Replans)", path=online_res["path"], secondary_path=online_res["initial_path"])
ax2.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## Task 6 — Performance Evaluation & Comparison Table

Finally, we benchmark all algorithms on the same potato-field map using path length, total crop-safe cost, near-crop risk steps, expanded nodes, execution time (ms), goal reachability, and replanning count.

In [ ]:
results = [
    bfs(grid, start, goal, penalty_weight=5),
    dfs(grid, start, goal, penalty_weight=5),
    greedy_best_first(grid, start, goal, penalty_weight=5),
    std_a,
    safe_a,
    online_res,
]

header = f"{'Algorithm':<22} {'Length':>8} {'Cost':>8} {'RiskSteps':>10} {'Expanded':>10} {'Time(ms)':>10} {'Reached':>9} {'Replans':>8}"
print(header)
print("-" * len(header))
for r in results:
    print(f"{r['name']:<22} {r['length']:>8} {r['cost']:>8} {r['risk_steps']:>10} {r['expanded']:>10} {r['time_ms']:>10.3f} {str(r['reached']):>9} {str(r.get('replans', '-')):>8}")

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, r in zip(axes.flat, results):
    render_grid_on_axis(ax, grid, f"{r['name']}\nLen: {r['length']} | Cost: {r['cost']} | Exp: {r['expanded']}", path=r["path"])
plt.tight_layout()
plt.show()